# 🤖 Notebook 5 — Sentiment Analysis and Urgency Scoring



                                                        


**Project:** AI-Driven Citizen Grievance Analysis

**Input:** data/processed/grievance_processed.csv ← from Notebook 2

**Output:** ../models/sentiment_model

### What this notebook does:
-  Ingest Cleaned Citizen Feedback from Week 2
-  Fine-tune a Transformer model (distilroberta-base — RoBERTa family)
-  Classify every grievance into 4 emotional categories

---

## 📋 4 Sentiment Classes

| Class | Label | Meaning |
|-------|-------|---------|
| 0 | Positive | Satisfied customers, praise, thanks |
| 1 | Neutral | Factual feedback, questions, observations |
| 2 | Negative | Complaints, frustration, issues |
| 3 | Critical/Urgent | Severe problems, system down, emergencies |

---

**Expected Accuracy:** ≥ 85%  
**Expected F1-Score:** ≥ 0.82

## Install / Upgrade Dependencies

Run this cell once, then **restart the kernel**, then run all cells from Cell 1 onwards.

In [1]:
# ── Install / upgrade required packages ──────────────────────────────────────
# Run this cell once, then RESTART the kernel before continuing
#import subprocess
#subprocess.run(['pip', 'install', '-q', '--upgrade', 'accelerate>=1.1.0'], check=True)
#print('✅ accelerate installed/upgraded')
#print('⚠️  Please RESTART the kernel now, then run from Cell 1 onwards.')

In [2]:
#upgrade
#pip install --upgrade numpy scipy

##  Import Libraries

In [2]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score,
    confusion_matrix, classification_report
)

# HuggingFace Transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported successfully')
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

✅ All libraries imported successfully
PyTorch version : 2.1.1+cpu
CUDA available  : False


## Configuration

In [3]:
# ── File paths ───────────────────────────────────────────────────────────────
INPUT_FILE      = '../data/processed/grievance_processed.csv'
MODEL_DIR       = '../models/'
DATA_OUTPUT_DIR = '../data/processed/'

# Ensure directories exist
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(DATA_OUTPUT_DIR, exist_ok=True)

# ── All settings in one dictionary ──────────────────────────────────────────
# Fast training defaults: short civic complaints rarely need 128 tokens.
CONFIG = {
    # Model
    'model_name'       : 'distilroberta-base',
    'num_labels'       : 4,
    'max_length'       : 64,

    # Training
    'train_batch_size' : 32 if torch.cuda.is_available() else 8,
    'eval_batch_size'  : 32 if torch.cuda.is_available() else 8,
    'num_epochs'       : 2,
    'learning_rate'    : 2e-5,
    'weight_decay'     : 0.01,
    'warmup_ratio'     : 0.1,
    'fp16'             : torch.cuda.is_available(),
    'early_stopping_patience' : 1,

    # Paths
    'output_dir'       : os.path.join(MODEL_DIR, 'sentiment_model'),

    # Device
    'device'           : 'cuda' if torch.cuda.is_available() else 'cpu'
}

# ── 4 Sentiment class labels ─────────────────────────────────────────────────
LABEL_MAP = {
    0 : 'positive',
    1 : 'neutral',
    2 : 'negative',
    3 : 'critical'
}

# Reverse map — used when loading from CSV
LABEL_MAP_REVERSE = {v: k for k, v in LABEL_MAP.items()}

# Create output directory
os.makedirs(CONFIG['output_dir'], exist_ok=True)

print('✅ Configuration loaded')
print(f'Input file : {INPUT_FILE}')
print(f'Model dir  : {MODEL_DIR}')
print(f'Output dir : {CONFIG["output_dir"]}')
print(f"Device     : {CONFIG['device']}")
print(f"Model      : {CONFIG['model_name']}")
print(f"Classes    : {LABEL_MAP}")
print(f"Fast mode  : {CONFIG['num_epochs']} epochs, max {CONFIG['max_length']} tokens, batch {CONFIG['train_batch_size']}, FP16={CONFIG['fp16']}")
if CONFIG['device'] == 'cpu':
    print('⚠️ CUDA is unavailable; training will be much slower on CPU. Install a CUDA-enabled PyTorch build to use the GPU.')

✅ Configuration loaded
Input file : ../data/processed/grievance_processed.csv
Model dir  : ../models/
Output dir : ../models/sentiment_model
Device     : cpu
Model      : distilroberta-base
Classes    : {0: 'positive', 1: 'neutral', 2: 'negative', 3: 'critical'}
Fast mode  : 2 epochs, max 64 tokens, batch 8, FP16=False
⚠️ CUDA is unavailable; training will be much slower on CPU. Install a CUDA-enabled PyTorch build to use the GPU.


##  Load Processed Data

Loads the real processed grievance data from Week 2.  
Requires `clean_text` column. If `sentiment_label` is missing, auto-labels using keyword rules.

In [4]:
# ── Load the refined dataset ────────────────────────────────────────────────
print(f'[INFO] Loading data from: {INPUT_FILE}')
df = pd.read_csv(INPUT_FILE)

# Basic sanity check
print(f'[INFO] Loaded {len(df):,} rows from {INPUT_FILE}')
print("Data Shape:", df.shape)
print(df[['clean_text']].head())

# Drop any rows where text processing failed
df = df.dropna(subset=['clean_text']).copy()
df['clean_text'] = df['clean_text'].astype(str)

# ── If sentiment_label missing — create using keyword rules ─────────────────
if 'sentiment_label' not in df.columns:
    print('[INFO] sentiment_label column missing — auto-labelling using keywords')

    critical_kw = ['urgent', 'emergency', 'collapse', 'fire', 'flood', 'danger',
                   'explosion', 'leak', 'trapped', 'immediately', 'life risk']
    negative_kw = ['broken', 'not working', 'issue', 'problem', 'complaint',
                   'no response', 'damage', 'poor', 'bad', 'failed', 'blocked']
    positive_kw = ['thank', 'thanks', 'great', 'excellent', 'appreciate', 'happy',
                   'good work', 'resolved', 'improved', 'wonderful']

    def auto_label(text):
        t = str(text).lower()
        if any(k in t for k in critical_kw):
            return 'critical'
        if any(k in t for k in negative_kw):
            return 'negative'
        if any(k in t for k in positive_kw):
            return 'positive'
        return 'neutral'

    df['sentiment_label'] = df['clean_text'].apply(auto_label)
    print('[INFO] Auto-labelling complete — review labels before using in production')

# Drop rows with unknown/null sentiment label
df = df.dropna(subset=['sentiment_label']).copy()

# ── Convert text labels to numbers ──────────────────────────────────────────
df['label'] = df['sentiment_label'].str.lower().map(LABEL_MAP_REVERSE)
df = df.dropna(subset=['label']).copy()
df['label'] = df['label'].astype(int)

# ── Summary ──────────────────────────────────────────────────────────────────
label_counts = df['label'].value_counts().reindex(LABEL_MAP.keys(), fill_value=0)
print(f'\n✅ Data ready — {len(df):,} clean rows')
print('\nLabel distribution:')
for lbl, name in LABEL_MAP.items():
    count = int(label_counts[lbl])
    pct   = count / len(df) * 100
    bar   = '█' * int(pct // 3)
    print(f'  Class {lbl} {name:10s}: {count:4d} ({pct:5.1f}%)  {bar}')

missing_labels = [LABEL_MAP[lbl] for lbl, count in label_counts.items() if count == 0]
if missing_labels:
    print(f'⚠️ Missing classes: {missing_labels}. Metrics for these classes cannot be learned until labeled examples are added.')

print(f'\nSample rows:')
print(df[['clean_text', 'sentiment_label', 'label']].head(6).to_string(index=False))

[INFO] Loading data from: ../data/processed/grievance_processed.csv
[INFO] Loaded 24,774 rows from ../data/processed/grievance_processed.csv
Data Shape: (24774, 24)
                                          clean_text
0                 public notice area violation issue
1                area situation control satisfactory
2     community update everything appears good order
3     public notice area system functioning properly
4  derelict vehicle license plate reviewed additi...

✅ Data ready — 24,774 clean rows

Label distribution:
  Class 0 positive  :    0 (  0.0%)  
  Class 1 neutral   : 13900 ( 56.1%)  ██████████████████
  Class 2 negative  : 8512 ( 34.4%)  ███████████
  Class 3 critical  : 2362 (  9.5%)  ███
⚠️ Missing classes: ['positive']. Metrics for these classes cannot be learned until labeled examples are added.

Sample rows:
                                        clean_text sentiment_label  label
                public notice area violation issue        negative      2
   

##  Train / Validation / Test Split

- **70% Train** — model learns from this
- **15% Validation** — checked after every epoch
- **15% Test** — final hidden exam, never seen during training

`stratify=` ensures all 4 classes have equal % in every split.

In [5]:
texts  = df['clean_text'].tolist()
labels = df['label'].tolist()

# ── Step 1: split off 30% as temp (will become val + test) ──────────────────
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts, labels,
    test_size    = 0.30,
    random_state = 42,
    stratify     = labels
)

# ── Step 2: split temp into 50/50 → val and test (15% each) ─────────────────
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels,
    test_size    = 0.50,
    random_state = 42,
    stratify     = temp_labels
)

print('✅ Data split completed')
print(f'  Train      : {len(train_texts):,} samples (70%)')
print(f'  Validation : {len(val_texts):,} samples (15%)')
print(f'  Test       : {len(test_texts):,} samples (15%)')

# ── Check which of the 4 classes are present in the test set ────────────────
test_class_counts = Counter(test_labels)
print(f'\n  Test set class distribution: {dict(sorted(test_class_counts.items()))}')

for i in range(CONFIG['num_labels']):
    name = LABEL_MAP[i]
    if i in test_class_counts:
        print(f'  [INFO] Class "{name}" found ✅')
    else:
        print(f'  [INFO] Class "{name}" not found — will be skipped in report')

✅ Data split completed
  Train      : 17,341 samples (70%)
  Validation : 3,716 samples (15%)
  Test       : 3,717 samples (15%)

  Test set class distribution: {1: 2085, 2: 1277, 3: 355}
  [INFO] Class "positive" not found — will be skipped in report
  [INFO] Class "neutral" found ✅
  [INFO] Class "negative" found ✅
  [INFO] Class "critical" found ✅


##  Tokenization

RoBERTa cannot read raw text — it needs numbers.  
The tokenizer converts text into token IDs.

**Note:** We use a custom `GrievanceDataset` class instead of `TensorDataset`.  
HuggingFace `Trainer` requires `__getitem__` to return a **dict** with keys  
`input_ids`, `attention_mask`, `labels` — `TensorDataset` returns a tuple which breaks the Trainer.

In [6]:
# ── Load tokenizer from HuggingFace ─────────────────────────────────────────
print(f"[INFO] Loading tokenizer: {CONFIG['model_name']}")
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

# ── HuggingFace-compatible Dataset class ────────────────────────────────────
# Dynamic padding avoids padding every sample to max_length before batching.
class GrievanceDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            texts,
            padding        = False,
            truncation     = True,
            max_length     = max_length
        )
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids'      : torch.tensor(self.encodings['input_ids'][idx], dtype=torch.long),
            'attention_mask' : torch.tensor(self.encodings['attention_mask'][idx], dtype=torch.long),
            'labels'         : torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ── Build datasets ──────────────────────────────────────────────────────────
print('[INFO] Tokenizing train / val / test splits...')
train_dataset = GrievanceDataset(train_texts, train_labels, tokenizer, CONFIG['max_length'])
val_dataset   = GrievanceDataset(val_texts,   val_labels,   tokenizer, CONFIG['max_length'])
test_dataset  = GrievanceDataset(test_texts,  test_labels,  tokenizer, CONFIG['max_length'])
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if CONFIG['device'] == 'cuda' else None
)

print('✅ Tokenization completed')
print(f'  Train dataset : {len(train_dataset):,} samples')
print(f'  Val dataset   : {len(val_dataset):,} samples')
print(f'  Test dataset  : {len(test_dataset):,} samples')

[INFO] Loading tokenizer: distilroberta-base
[INFO] Tokenizing train / val / test splits...
✅ Tokenization completed
  Train dataset : 17,341 samples
  Val dataset   : 3,716 samples
  Test dataset  : 3,717 samples


##  Load distilroberta-base Model

- Pre-trained on 160GB of text — already understands language deeply
- We add a **classification head** (4 output neurons) on top and fine-tune
- `num_labels=4` → one output per sentiment class

In [7]:
# ── Load pre-trained distilroberta with 4-class classification head ──────────
print(f"[INFO] Loading model: {CONFIG['model_name']}")

model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG['model_name'],
    num_labels              = CONFIG['num_labels'],
    id2label                = {str(k): v for k, v in LABEL_MAP.items()},
    label2id                = {v: str(k) for k, v in LABEL_MAP.items()},
    ignore_mismatched_sizes = True
)

# Move model to GPU if available
model = model.to(CONFIG['device'])

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'✅ Model loaded: {CONFIG["model_name"]}')
print(f'  Total parameters     : {total_params:,}')
print(f'  Trainable parameters : {trainable_params:,}')
print(f'  Running on           : {CONFIG["device"]}')

[INFO] Loading model: distilroberta-base


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.weight', 'classifier.out_proj.weight', 'classifier.dense.bias', 'classifier.out_proj.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model loaded: distilroberta-base
  Total parameters     : 82,121,476
  Trainable parameters : 82,121,476
  Running on           : cpu


## Metrics Function

Called by Trainer after every epoch to measure:
- **Accuracy** — % of grievances classified correctly
- **F1 (weighted)** — balances precision and recall for all 4 classes

In [8]:
def compute_metrics(eval_pred):
    """Calculate accuracy plus macro and weighted F1."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    return {
        'accuracy'    : round(accuracy_score(labels, predictions), 4),
        'f1_macro'    : round(f1_score(labels, predictions, average='macro', zero_division=0), 4),
        'f1_weighted' : round(f1_score(labels, predictions, average='weighted', zero_division=0), 4)
    }

print('✅ Metrics function defined')
print('   Tracks: accuracy + macro F1 + weighted F1')

✅ Metrics function defined
   Tracks: accuracy + macro F1 + weighted F1


##  Fine-tune the Model (Optimized Training)

- Dynamic padding reduces unnecessary computation per batch.
- Length-grouped batches reduce padding waste further.
- Class-weighted loss improves recall for minority classes.
- Early stopping ends training when macro F1 stops improving.
- `evaluation_strategy` is used for Transformers 4.35 compatibility.

In [9]:
# ── Compute warmup_steps from warmup_ratio ───────────────────────────────────
steps_per_epoch = max(1, len(train_dataset) // CONFIG['train_batch_size'])
total_steps     = steps_per_epoch * CONFIG['num_epochs']
warmup_steps    = int(CONFIG['warmup_ratio'] * total_steps)

# Inverse-frequency weights improve recall for present minority classes.
train_counts = Counter(train_labels)
class_weights = torch.zeros(CONFIG['num_labels'], dtype=torch.float)
for label_id in range(CONFIG['num_labels']):
    count = train_counts.get(label_id, 0)
    class_weights[label_id] = (
        len(train_labels) / (CONFIG['num_labels'] * count)
        if count else 0.0
    )

print(f'  Steps per epoch : {steps_per_epoch}')
print(f'  Total steps     : {total_steps}')
print(f'  Warmup steps    : {warmup_steps} ({CONFIG["warmup_ratio"]*100:.0f}% of total)')
print(f'  Class weights   : {class_weights.tolist()}')

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights.to(model.device))
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

# ── Training arguments ───────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir                  = CONFIG['output_dir'],
    num_train_epochs            = CONFIG['num_epochs'],
    per_device_train_batch_size = CONFIG['train_batch_size'],
    per_device_eval_batch_size  = CONFIG['eval_batch_size'],
    learning_rate               = CONFIG['learning_rate'],
    weight_decay                = CONFIG['weight_decay'],
    warmup_steps                = warmup_steps,
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1_macro',
    greater_is_better            = True,
    logging_steps               = 10,
    save_total_limit            = 2,
    report_to                   = 'none',
    fp16                        = CONFIG['fp16'],
    group_by_length             = True,
)

# ── Trainer — bundles model + data + training config ────────────────────────
trainer = WeightedTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
    data_collator   = data_collator,
    callbacks       = [EarlyStoppingCallback(
        early_stopping_patience=CONFIG['early_stopping_patience']
    )]
)

# ── Train ─────────────────────────────────────────────────────────────────────
print('\n🔄 Starting fine-tuning...')
print(f'   Model   : {CONFIG["model_name"]}')
print(f'   Epochs  : {CONFIG["num_epochs"]} (early stopping enabled)')
print(f'   Device  : {CONFIG["device"]}')
print(f'   Train   : {len(train_dataset):,} samples')
print(f'   Val     : {len(val_dataset):,} samples')
print()

train_result = trainer.train()

print(f'\n✅ Fine-tuning completed!')
print(f'   Training loss : {train_result.training_loss:.4f}')
print(f'   Total steps   : {train_result.global_step}')

  Steps per epoch : 2167
  Total steps     : 4334
  Warmup steps    : 433 (10% of total)
  Class weights   : [0.0, 0.4455549716949463, 0.7276350855827332, 2.6226558685302734]

🔄 Starting fine-tuning...
   Model   : distilroberta-base
   Epochs  : 2 (early stopping enabled)
   Device  : cpu
   Train   : 17,341 samples
   Val     : 3,716 samples



  0%|          | 0/4336 [00:00<?, ?it/s]

You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


{'loss': 1.4406, 'learning_rate': 4.618937644341802e-07, 'epoch': 0.0}
{'loss': 1.32, 'learning_rate': 9.237875288683604e-07, 'epoch': 0.01}
{'loss': 1.2198, 'learning_rate': 1.3856812933025404e-06, 'epoch': 0.01}
{'loss': 1.372, 'learning_rate': 1.8475750577367208e-06, 'epoch': 0.02}
{'loss': 1.4686, 'learning_rate': 2.3094688221709006e-06, 'epoch': 0.02}
{'loss': 1.4127, 'learning_rate': 2.771362586605081e-06, 'epoch': 0.03}
{'loss': 1.2676, 'learning_rate': 3.233256351039261e-06, 'epoch': 0.03}
{'loss': 1.1275, 'learning_rate': 3.6951501154734416e-06, 'epoch': 0.04}
{'loss': 1.1823, 'learning_rate': 4.157043879907622e-06, 'epoch': 0.04}
{'loss': 1.1023, 'learning_rate': 4.618937644341801e-06, 'epoch': 0.05}
{'loss': 1.6115, 'learning_rate': 5.080831408775982e-06, 'epoch': 0.05}
{'loss': 0.8097, 'learning_rate': 5.542725173210162e-06, 'epoch': 0.06}
{'loss': 0.6794, 'learning_rate': 6.004618937644342e-06, 'epoch': 0.06}
{'loss': 0.6156, 'learning_rate': 6.466512702078522e-06, 'epoch'

  0%|          | 0/465 [00:00<?, ?it/s]

{'eval_loss': 8.630154479760677e-05, 'eval_accuracy': 1.0, 'eval_f1_macro': 1.0, 'eval_f1_weighted': 1.0, 'eval_runtime': 48.7065, 'eval_samples_per_second': 76.294, 'eval_steps_per_second': 9.547, 'epoch': 1.0}
{'loss': 0.0001, 'learning_rate': 1.1099154496541123e-05, 'epoch': 1.0}
{'loss': 0.0001, 'learning_rate': 1.1047911862669743e-05, 'epoch': 1.01}
{'loss': 0.0001, 'learning_rate': 1.099666922879836e-05, 'epoch': 1.01}
{'loss': 0.0001, 'learning_rate': 1.094542659492698e-05, 'epoch': 1.01}
{'loss': 0.0001, 'learning_rate': 1.08941839610556e-05, 'epoch': 1.02}
{'loss': 0.0001, 'learning_rate': 1.0842941327184217e-05, 'epoch': 1.02}
{'loss': 0.0001, 'learning_rate': 1.0791698693312837e-05, 'epoch': 1.03}
{'loss': 0.0001, 'learning_rate': 1.0740456059441455e-05, 'epoch': 1.03}
{'loss': 0.0001, 'learning_rate': 1.0689213425570076e-05, 'epoch': 1.04}
{'loss': 0.0001, 'learning_rate': 1.0637970791698696e-05, 'epoch': 1.04}
{'loss': 0.0001, 'learning_rate': 1.0586728157827313e-05, 'epoc

  0%|          | 0/465 [00:00<?, ?it/s]

{'eval_loss': 4.784122575074434e-05, 'eval_accuracy': 1.0, 'eval_f1_macro': 1.0, 'eval_f1_weighted': 1.0, 'eval_runtime': 58.0119, 'eval_samples_per_second': 64.056, 'eval_steps_per_second': 8.016, 'epoch': 2.0}
{'train_runtime': 4399.6303, 'train_samples_per_second': 7.883, 'train_steps_per_second': 0.986, 'train_loss': 0.051305586601020005, 'epoch': 2.0}

✅ Fine-tuning completed!
   Training loss : 0.0513
   Total steps   : 4336


##  Evaluate on Test Set

Test the model on the **15% hidden test data** it never saw during training.

- **Overall accuracy** and **F1 score**
- **Classification report** — precision, recall, F1 per class
- **Confusion matrix** — which classes get confused with each other

In [10]:
# ── Run predictions on test set ──────────────────────────────────────────────
print('🔄 Evaluating on hidden test set...')
test_output  = trainer.predict(test_dataset)
pred_labels  = np.argmax(test_output.predictions, axis=1)
true_labels  = test_labels

# ── Core metrics ─────────────────────────────────────────────────────────────
accuracy    = accuracy_score(true_labels, pred_labels)
f1_macro    = f1_score(true_labels, pred_labels, average='macro', zero_division=0)
f1_weighted = f1_score(true_labels, pred_labels, average='weighted', zero_division=0)
f1          = f1_macro

print(f'\n{"="*65}')
print(f'  STAGE 1 — TEST SET RESULTS')
print(f'{"="*65}')
print(f'  Accuracy       : {accuracy:.4f}  (target ≥ 0.85)')
print(f'  Macro F1       : {f1_macro:.4f}  (primary metric)')
print(f'  Weighted F1    : {f1_weighted:.4f}')
print(f'  Status         : {"✅ PASS" if accuracy >= 0.85 and f1_macro >= 0.82 else "⚠️ Below target"}')
print(f'{"="*65}')

# ── Per-class classification report ─────────────────────────────────────────
print('\n📊 Classification Report (per class):')
print(classification_report(
    true_labels,
    pred_labels,
    labels        = [0, 1, 2, 3],
    target_names  = [LABEL_MAP[i] for i in range(4)],
    zero_division = 0
))

# ── Confusion matrix ─────────────────────────────────────────────────────────
cm = confusion_matrix(true_labels, pred_labels, labels=[0, 1, 2, 3])
print('🎯 Confusion Matrix:')
print(f'  Rows = True class | Columns = Predicted class')
print(f'  Classes: {[LABEL_MAP[i] for i in range(4)]}')
print(cm)

🔄 Evaluating on hidden test set...


  0%|          | 0/465 [00:00<?, ?it/s]


  STAGE 1 — TEST SET RESULTS
  Accuracy       : 1.0000  (target ≥ 0.85)
  Macro F1       : 1.0000  (primary metric)
  Weighted F1    : 1.0000
  Status         : ✅ PASS

📊 Classification Report (per class):
              precision    recall  f1-score   support

    positive       0.00      0.00      0.00         0
     neutral       1.00      1.00      1.00      2085
    negative       1.00      1.00      1.00      1277
    critical       1.00      1.00      1.00       355

   micro avg       1.00      1.00      1.00      3717
   macro avg       0.75      0.75      0.75      3717
weighted avg       1.00      1.00      1.00      3717

🎯 Confusion Matrix:
  Rows = True class | Columns = Predicted class
  Classes: ['positive', 'neutral', 'negative', 'critical']
[[   0    0    0    0]
 [   0 2085    0    0]
 [   0    0 1277    0]
 [   0    0    0  355]]


##  Save Trained Model

Saves everything needed for future inference:
- `model` → trained weights
- `tokenizer` → vocabulary and encoding rules
- `stage1_results.json` → accuracy, F1, label map

In [11]:
# ── Save model + tokenizer ───────────────────────────────────────────────────
trainer.save_model(CONFIG['output_dir'])
model.save_pretrained(CONFIG['output_dir'])
tokenizer.save_pretrained(CONFIG['output_dir'])

print(f"✅ Model saved to    : {CONFIG['output_dir']}")
print(f"✅ Tokenizer saved to: {CONFIG['output_dir']}")

# ── Save config.json with results ────────────────────────────────────────────
results_config = {
    'model_name'   : CONFIG['model_name'],
    'num_labels'   : CONFIG['num_labels'],
    'label_map'    : LABEL_MAP,
    'accuracy'     : round(float(accuracy), 4),
    'f1_macro'     : round(float(f1_macro), 4),
    'f1_weighted'  : round(float(f1_weighted), 4),
    'max_length'   : CONFIG['max_length'],
    'device'       : CONFIG['device']
}

config_path = f"{CONFIG['output_dir']}/stage1_results.json"
with open(config_path, 'w') as f:
    json.dump(results_config, f, indent=2)

print(f'✅ Results saved to  : {config_path}')
print(f'\nSaved config:')
print(json.dumps(results_config, indent=2))

✅ Model saved to    : ../models/sentiment_model
✅ Tokenizer saved to: ../models/sentiment_model
✅ Results saved to  : ../models/sentiment_model/stage1_results.json

Saved config:
{
  "model_name": "distilroberta-base",
  "num_labels": 4,
  "label_map": {
    "0": "positive",
    "1": "neutral",
    "2": "negative",
    "3": "critical"
  },
  "accuracy": 1.0,
  "f1_macro": 1.0,
  "f1_weighted": 1.0,
  "max_length": 64,
  "device": "cpu"
}


## Live Inference on New Grievances

Loads the saved model and classifies ANY new complaint text instantly.

In [12]:
# ── Load saved model for inference ──────────────────────────────────────────
loaded_tokenizer = AutoTokenizer.from_pretrained(CONFIG['output_dir'])
loaded_model     = AutoModelForSequenceClassification.from_pretrained(CONFIG['output_dir'])
loaded_model     = loaded_model.to(CONFIG['device'])
loaded_model.eval()

print('✅ Model loaded for inference')

# ── Inference function ───────────────────────────────────────────────────────
def classify_grievance(text: str) -> dict:
    """
    Classify a single citizen grievance into one of 4 sentiment classes.

    Returns:
        dict with sentiment label, confidence %, and all class probabilities
    """
    inputs = loaded_tokenizer(
        text,
        return_tensors = 'pt',
        truncation     = True,
        padding        = 'max_length',
        max_length     = CONFIG['max_length']
    )

    inputs = {k: v.to(CONFIG['device']) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = loaded_model(**inputs)

    probs      = torch.softmax(outputs.logits, dim=-1)[0]
    pred_label = torch.argmax(probs).item()
    confidence = probs[pred_label].item()

    return {
        'text'       : text,
        'sentiment'  : LABEL_MAP[pred_label],
        'class_id'   : pred_label,
        'confidence' : round(confidence * 100, 2),
        'all_probs'  : {
            LABEL_MAP[i]: round(probs[i].item() * 100, 2)
            for i in range(4)
        }
    }

# ── Test on 8 real citizen grievances ────────────────────────────────────────
test_grievances = [
    'The road repair near school was done beautifully. Thank you so much!',
    'When will the water supply schedule be announced for our area?',
    'Garbage has not been collected from our street for 6 days.',
    'Live electric wire on the road after the storm. Someone may die!',
    'The new park benches installed are very comfortable. Great initiative.',
    'The drainage pipe near the market is completely blocked.',
    'What are the timings of the municipal office on Saturdays?',
    'Building collapsed in sector 5. People are trapped. Send help now!',
]

print(f'\n{"="*75}')
print('  STAGE 1 OUTPUT — GRIEVANCE SENTIMENT CLASSIFICATION')
print(f'{"="*75}')

for i, grievance in enumerate(test_grievances, 1):
    result = classify_grievance(grievance)
    emoji  = {'positive':'😊','neutral':'😐','negative':'😠','critical':'🚨'}[result['sentiment']]
    print(f'\n[{i}] {result["text"][:65]}...' if len(result['text']) > 65 else f'\n[{i}] {result["text"]}')
    print(f'    Sentiment  : {emoji}  {result["sentiment"].upper()}  ({result["confidence"]}% confidence)')
    print(f'    All scores : ', end='')
    for cls, prob in result['all_probs'].items():
        print(f'{cls}={prob}%', end='  ')
    print()

print(f'\n{"="*75}')

✅ Model loaded for inference

  STAGE 1 OUTPUT — GRIEVANCE SENTIMENT CLASSIFICATION

[1] The road repair near school was done beautifully. Thank you so mu...
    Sentiment  : 😐  NEUTRAL  (99.99% confidence)
    All scores : positive=0.0%  neutral=99.99%  negative=0.0%  critical=0.0%  

[2] When will the water supply schedule be announced for our area?
    Sentiment  : 😐  NEUTRAL  (99.99% confidence)
    All scores : positive=0.0%  neutral=99.99%  negative=0.0%  critical=0.01%  

[3] Garbage has not been collected from our street for 6 days.
    Sentiment  : 😐  NEUTRAL  (99.98% confidence)
    All scores : positive=0.0%  neutral=99.98%  negative=0.01%  critical=0.0%  

[4] Live electric wire on the road after the storm. Someone may die!
    Sentiment  : 😐  NEUTRAL  (99.99% confidence)
    All scores : positive=0.0%  neutral=99.99%  negative=0.0%  critical=0.01%  

[5] The new park benches installed are very comfortable. Great initia...
    Sentiment  : 😐  NEUTRAL  (99.99% confidence)
  

In [13]:
# ── FULL DATASET INFERENCE — Run model on all grievances ────────────────────────
# Batch inference avoids one tokenizer/model call per row.
print('\nRunning full-dataset inference...')
print('This will add sentiment_label and model_confidence to grievance_processed.csv')

from tqdm import tqdm

# ── Define vectorized batch inference ────────────────────────────────────────
def infer_batch(texts_batch: list) -> list:
    """Classify a batch of grievances in one model call."""
    inputs = loaded_tokenizer(
        texts_batch,
        return_tensors = 'pt',
        truncation     = True,
        padding        = True,
        max_length     = CONFIG['max_length']
    )
    inputs = {key: value.to(CONFIG['device']) for key, value in inputs.items()}

    with torch.no_grad():
        logits = loaded_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)

    pred_labels = torch.argmax(probs, dim=1).tolist()
    confidences = probs.max(dim=1).values.tolist()
    return [
        {
            'sentiment_label': LABEL_MAP[label_id],
            'model_confidence': round(confidence * 100, 2)
        }
        for label_id, confidence in zip(pred_labels, confidences)
    ]

# ── Run inference on full dataset ───────────────────────────────────────────
print(f'Classifying {len(df):,} grievances...\n')
batch_size = 128 if CONFIG['device'] == 'cuda' else 32
all_results = []

for batch_start in tqdm(range(0, len(df), batch_size), desc='Processing batches'):
    batch_end = min(batch_start + batch_size, len(df))
    batch_texts = df['clean_text'].iloc[batch_start:batch_end].tolist()
    all_results.extend(infer_batch(batch_texts))

# ── Add results to dataframe ─────────────────────────────────────────────────
df['sentiment_label'] = [result['sentiment_label'] for result in all_results]
df['model_confidence'] = [result['model_confidence'] for result in all_results]

print(f'\n✅ Inference complete on {len(df):,} grievances')
print('\nSentiment distribution:')
print(df['sentiment_label'].value_counts())
print('\nConfidence stats:')
print(df['model_confidence'].describe())

# ── Save updated dataframe with sentiment predictions ────────────────────────
output_file = INPUT_FILE
df.to_csv(output_file, index=False)

print(f'\n✅ Saved to: {output_file}')
print(f'   Columns: {list(df.columns)}')
print(f'   Shape: {df.shape}')
print('\n✅ Ready for Notebook 6 (Urgency Scoring)')


Running full-dataset inference...
This will add sentiment_label and model_confidence to grievance_processed.csv
Classifying 24,774 grievances...



Processing batches: 100%|██████████| 775/775 [08:12<00:00,  1.57it/s]



✅ Inference complete on 24,774 grievances

Sentiment distribution:
neutral     13899
negative     8512
critical     2363
Name: sentiment_label, dtype: int64

Confidence stats:
count    24774.000000
mean        99.991767
std          0.005912
min         99.830000
25%         99.990000
50%         99.990000
75%        100.000000
max        100.000000
Name: model_confidence, dtype: float64

✅ Saved to: ../data/processed/grievance_processed.csv
   Columns: ['Unique Key', 'Created Date', 'Closed Date', 'Agency Name', 'Complaint Type', 'Descriptor', 'Location Type', 'Borough', 'Status', 'Resolution Description', 'is_complaint', 'resolution_hours', 'hour_created', 'day_of_week', 'closed_date_fmt', 'resolution_hours_fmt', 'hour_created_fmt', 'combined_text', 'clean_text', 'token_count', 'unique_tokens', 'sentiment_label', 'label', 'model_confidence']
   Shape: (24774, 24)

✅ Ready for Notebook 6 (Urgency Scoring)


##  Stage 1 Summary

In [14]:
print('=' * 65)
print('  STAGE 1 COMPLETE — SUMMARY')
print('=' * 65)

_accuracy = globals().get('accuracy', 0.0)
_f1_macro = globals().get('f1_macro', 0.0)
_f1_weighted = globals().get('f1_weighted', 0.0)

checklist = [
    ('Ingest Cleaned Citizen Feedback from Week 2',
     True,
     'Loaded from grievance_processed.csv'),
    ('Fine-tune Transformer with optimized batching',
     True,
     f'{CONFIG["model_name"]} — dynamic padding, grouped batches, early stopping'),
    ('Classify into 4 emotional categories',
     len(set(labels)) == CONFIG['num_labels'],
     'Positive / Neutral / Negative / Critical'),
    ('Train / Validation / Test split with stratification',
     True,
     '70% train / 15% val / 15% test — stratified'),
    ('Evaluate — Accuracy + Macro F1',
     _accuracy >= 0.85 and _f1_macro >= 0.82,
     f'Accuracy={_accuracy:.4f} | Macro F1={_f1_macro:.4f}'),
    ('Classification report + confusion matrix',
     True,
     'Printed with labels=[0,1,2,3]'),
    ('Save model + tokenizer + config',
     True,
     f'Saved to {CONFIG["output_dir"]}'),
    ('Fast batched inference on all grievances',
     True,
     'Uses one model call per batch'),
]

for req, done, note in checklist:
    icon = '✅' if done else '⚠️'
    print(f'  {icon}  {req}')
    print(f'       → {note}')

print()
print(f'  Final Accuracy       : {_accuracy:.4f}')
print(f'  Final Macro F1       : {_f1_macro:.4f}')
print(f'  Final Weighted F1    : {_f1_weighted:.4f}')
print(f'  Model saved to       : {CONFIG["output_dir"]}')
print()
print('  Next → Stage 2: Urgency Scoring & Priority Assignment')
print('=' * 65)

  STAGE 1 COMPLETE — SUMMARY
  ✅  Ingest Cleaned Citizen Feedback from Week 2
       → Loaded from grievance_processed.csv
  ✅  Fine-tune Transformer with optimized batching
       → distilroberta-base — dynamic padding, grouped batches, early stopping
  ⚠️  Classify into 4 emotional categories
       → Positive / Neutral / Negative / Critical
  ✅  Train / Validation / Test split with stratification
       → 70% train / 15% val / 15% test — stratified
  ✅  Evaluate — Accuracy + Macro F1
       → Accuracy=1.0000 | Macro F1=1.0000
  ✅  Classification report + confusion matrix
       → Printed with labels=[0,1,2,3]
  ✅  Save model + tokenizer + config
       → Saved to ../models/sentiment_model
  ✅  Fast batched inference on all grievances
       → Uses one model call per batch

  Final Accuracy       : 1.0000
  Final Macro F1       : 1.0000
  Final Weighted F1    : 1.0000
  Model saved to       : ../models/sentiment_model

  Next → Stage 2: Urgency Scoring & Priority Assignment
